# Notebook for trt selection development

Tasks:
- LOW PRIORITY: add supportive handling when supportives are optional
- combine source phases into combination phases where specified
- move to protocol level, subset phases selection off studies <- NEXT STEP
- create output objects and populate, sense-check phases, push as first pass. 
- improve description module, move sig descriptions to hemonc_import?

In [1]:
from pathlib import Path
import os
import sys
import yaml

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

In [2]:
REGISTRY_PATH = ROOT / "scoop" / "configuration" / "profiles" / "nsclc" / "nsclc_protocol_registry.yaml"

In [3]:
with REGISTRY_PATH.open("r", encoding="utf-8") as handle:
    raw = yaml.safe_load(handle)

In [4]:
raw["nsclc"].keys()

dict_keys(['version', 'conditions', 'source_phases', 'phase_combinations', 'protocols', 'entries'])

In [5]:
import scoop.configuration.protocol_registry as protocol_registry

#### First layer - read phase sources, and combination data

In [6]:
default_conditions = raw["nsclc"]["conditions"]

In [7]:
default_conditions

['Lung cancer', 'Non-small cell lung cancer']

In [8]:
import sqlalchemy.orm as so
import sqlalchemy as sa
HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]
def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved
engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

In [9]:
from scoop.utils.hemonc_helpers import condition_cuis_by_names

In [10]:
test_input = ["Lung cancer"]

In [11]:
with so.Session(engine) as session:
    print(condition_cuis_by_names(test_input, session))

[46104]


In [12]:
source_phases = {}
for label, source_phase in raw["nsclc"]["source_phases"].items():
    with so.Session(engine) as session:
        source_phases[label]=protocol_registry.SourcePhaseYaml.generate(source_phase, label, default_conditions, session)

In [13]:
source_phases

{'surgery_only': SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), optional_reqs=None, curative_intent=True, regimens=None, type='definitive'),
 'port': SourcePhaseYaml(source_phase_id='port', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='min', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}), optional_reqs=None, curative_intent=True, regimens=['Radiation therapy'], type='adjuvant'),

In [14]:
raw["nsclc"]["protocols"]["neoadj_or_peri_ici_w_chemo"]

{'phases': ['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'],
 'optional_phases': {'adjuvant_ici': {'criteria': {'shared': 'component',
    'as_phase': 'neoadjuvant_chemo_ici'}}}}

In [15]:
protocols = {}
for label, protocol_data in raw["nsclc"]["protocols"].items():
    protocols[label] = protocol_registry.ProtocolYaml.generate(label, protocol_data)

In [16]:
protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], phases_for_removal=['adj_osimertinib', 'port'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], phases_for_removal=['port'], optional_phases=None),
 'neoadj_or_peri_ici_w_chemo': ProtocolYaml(protocol_id='neoadj_or_peri_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], phases_for_removal=None, optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], phases_for_removal=None, optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab'

In [17]:
phase_unions = {}
for label, union_data in raw["nsclc"]["phase_combinations"].items():
    phase_unions[label] = protocol_registry.PhaseUnionYaml(phase_id=label, 
                                                           source_phase_ids=union_data["source_phases"],
                                                           remove=union_data.get("remove", None))

In [18]:
phase_unions

{'met_intervention': PhaseUnionYaml(phase_id='met_intervention', source_phase_ids=['met_tki', 'met_ptdb', 'met_ptdb_w_ici'], remove=['met_tki', 'met_ptdb', 'met_ptdb_w_ici']),
 'ici_monotherapy_or_ici_with_chemotherapy_squamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_squamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_squamous'], remove=None),
 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_nonsquamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_nonsquamous'], remove=None),
 'chemotherapy_or_atezolizumab_squamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_squamous', source_phase_ids=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab'], remove=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab']),
 'chemotherapy_or_atezolizumab_nonsquamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_nonsquamous', source_phase_ids=['ptdb_nonsquamous'

#### 2nd layer - produce all protocols w/ categorised phases

In [19]:
phases_for_removal = set()

In [20]:
source_phases["surgery_only"].categorise()

{'surgery'}

In [21]:
def find_retrievability(broad_categories):
    if "chemotherapy" in broad_categories or "immunotherapy" in broad_categories:
        return "variantable"
    elif "radiotherapy" in broad_categories:
        return "rt"
    elif "surgery" in broad_categories:
        return "surgery"
    else:
        return "null"

In [22]:
import scoop.utils.phase_resolution as phase_resolution

In [23]:
eg_surgery_phase = source_phases["surgery_only"]

In [24]:
eg_surgery_phase

SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), optional_reqs=None, curative_intent=True, regimens=None, type='definitive')

In [25]:
phase_resolution.SurgeryPhase(phase_id=eg_surgery_phase.source_phase_id, phase_type=eg_surgery_phase.type, origin=eg_surgery_phase)

SurgeryPhase(phase_id='surgery_only', phase_type='definitive', origin=SourcePhaseYaml(source_phase_id='surgery_only', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='min', amount=1)}), optional_reqs=None, curative_intent=True, regimens=None, type='definitive'))

In [26]:
eg_variantable_phase = source_phases["neoadjuvant_chemo_ici"]
eg_variantable_phase

SourcePhaseYaml(source_phase_id='neoadjuvant_chemo_ici', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[['nivolumab', 'pembrolizumab', 'durvalumab', 'tislelizumab']], category_restrictions={'chemotherapy': CategoryRestriction(type='min', amount=1), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=1), 'surgery': CategoryRestriction(type='exact', amount=0)}), optional_reqs=None, curative_intent=True, regimens=None, type='neoadjuvant')

In [27]:
# phase_resolution.VariantablePhase()

In [28]:
for phase in source_phases.values():
    print(phase.source_phase_id)
    cats = phase.categorise()
    print(cats)
    print(find_retrievability(cats))
    print("")

surgery_only
{'surgery'}
surgery

port
{'radiotherapy'}
rt

adj_osimertinib
{'chemotherapy'}
variantable

neoadjuvant_chemo_ici
{'chemotherapy', 'immunotherapy'}
variantable

adjuvant_ici
{'chemotherapy'}
variantable

adjuvant_pembrolizumab
{'immunotherapy'}
variantable

adjuvant_atezolizumab_pembrolizumab
{'immunotherapy'}
variantable

adjuvant_osimertinib
{'chemotherapy'}
variantable

adjuvant_cisplatin_based
{'chemotherapy'}
variantable

adjuvant_optchemo_alectinib
{'chemotherapy'}
variantable

conc_crt
{'chemotherapy', 'radiotherapy'}
variantable

seq_crt_chemo
{'chemotherapy'}
variantable

definitive_rt
{'radiotherapy'}
rt

consolidation_durvalumab
{'immunotherapy'}
variantable

consolidation_osimertinib
{'chemotherapy'}
variantable

egfr_intervention
{'chemotherapy', 'immunotherapy'}
variantable

alk_intervention
{'chemotherapy'}
variantable

ros1_intervention
{'chemotherapy'}
variantable

braf_intervention
{'chemotherapy'}
variantable

ret_intervention
{'chemotherapy'}
variantab

In [29]:
from scoop.utils.variant_selection import build_variant_query_artifacts #, execute_variant_query


In [30]:
build_variant_query_artifacts(source_phases["neoadjuvant_chemo_ici"].component_requirements, 
                              source_phases["neoadjuvant_chemo_ici"].condition_cuis)

VariantQueryArtifacts(components_subquery=<sqlalchemy.sql.selectable.Select object at 0x7fb7b5512210>, categorised_components_subquery=<sqlalchemy.sql.selectable.Subquery at 0x7fb7b61dda90; anon_1>, variant_cuis_subquery=<sqlalchemy.sql.selectable.Subquery at 0x7fb7b5594180; anon_1>, variants_stmt=<sqlalchemy.sql.selectable.Select object at 0x7fb7b5513020>)

In [31]:
# engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))
# with so.Session(engine) as session:
#     variants = execute_variant_query(session, eg_variantable_phase)

In [32]:
# len(variants)

In [33]:
from scoop.utils.hemonc_helpers import _load_hemonc_models
models=_load_hemonc_models()

In [34]:
condition_cuis = [46104, 642]

## Working on new RT pathway

In [35]:
## Working on new RT pathway

In [36]:
rt_phases = []
for phase in source_phases.values():
    process_type = find_retrievability(phase.categorise())
    if process_type == "rt":
        rt_phases.append(phase)

In [37]:
for phase in rt_phases:
    print(phase)

source_phase_id='port' condition_cuis=[46104, 642] component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='min', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}) optional_reqs=None curative_intent=True regimens=['Radiation therapy'] type='adjuvant'
source_phase_id='definitive_rt' condition_cuis=[46104, 642] component_requirements=ComponentRequirements(component_lists=[[]], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=0), 'radiotherapy': CategoryRestriction(type='exact', amount=1), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0), 'supportive': CategoryRestriction(type='exact', amount=0)}) optional_reqs=None curative_intent=True regimens=['Radiation therapy'] type='defini

In [38]:
# if regimen, retrieve by regimen
# elif ComponentRequirements retrieve by component requirements
# else Exception

## Change of mind
# filter on both regimen and componentrequirements, making sure for both to be safe if permissive

#### Working on possible components replacement

In [39]:
import pandas as pd

In [40]:
from scoop.utils.variant_selection import build_components_subquery

In [41]:
with so.Session(engine) as session:
    original_test = pd.DataFrame(session.execute(sa.select(build_components_subquery(condition_cuis))).unique())

In [42]:
original_test

,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class
0,141786,Paclitaxel,379,paclitaxel,Taxane,Microtubule inhibitor,Taxane
1,141786,Carboplatin,88,carboplatin,Platinum agent,Alkylating agent,Platinum agent
2,141786,Bevacizumab,68,bevacizumab,Anti-VEGFR antibody,Antineoplastic monoclonal antibody,Anti-VEGFR antibody
3,141786,Atezolizumab,49,atezolizumab,Anti-PD-L1 antibody,Immunostimulatory monoclonal antibody,Anti-PD-L1 antibody
4,141787,Paclitaxel,379,paclitaxel,Taxane,Microtubule inhibitor,Taxane
...,...,...,...,...,...,...,...
827,161285,Durvalumab,173,durvalumab,Anti-PD-L1 antibody,Immunostimulatory monoclonal antibody,Anti-PD-L1 antibody
828,161307,Sunvozertinib,161113,NaN,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor
829,161308,Taletrectinib,161120,NaN,ROS1 inhibitor,NaN,NaN
830,161311,Zongertinib,161243,NaN,ERBB2 inhibitor,Tyrosine kinase inhibitor,NaN


ISSUE: UNIMPORTANT
- sig of variant cui lists FEWER studies than variant
- variant in question: cisplatin monotherapy 131009	

In [43]:
def var_stmt_from_cuilist(cuilist: list):
    return sa.select(models.Variants).where(models.Variants.variant_cui.in_(cuilist))

In [44]:
with so.Session(engine) as session:
    missing_vars = session.execute(var_stmt_from_cuilist([131009])).unique().scalars().all()

In [45]:
# this is the reason for the difference:
# sig study items different to variant study items
# despite being monotherapy
# how can the variant have been studied in that context without the
# corresponding sig also being studied? 
for study_item in missing_vars[0].study_items:
    print(study_item.study_objects[0].study)
    print(study_item.study_objects[0].condition)

GEFCAPI 02
Carcinoma of unknown primary
Gatzemeier et al. 2000
Non-small cell lung cancer
Jacobs et al. 1992
Head and neck cancer


In [46]:
missing_vars[0].component_sigs[0].study

'GEFCAPI 02|Jacobs et al. 1992'

##### Old troubleshooting

130670 is from a breast cancer study....
and 130863 from a nasopharyngeal carcinoma study

In [47]:
missing_varinfo_stmt = (
    sa.select(models.Variants).where(models.Variants.variant_cui.in_([130670, 130863]))
)

In [48]:
with so.Session(engine) as session:
    missing_vars = session.execute(missing_varinfo_stmt).unique().scalars().all()

In [49]:
for var in missing_vars:
    print(f"{var.variant_cui}")

130670
130670
130670
130863
130863
130863
130863


In [50]:
missing_vars[-1].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fb7b5204a70>,
 'allsigshavefrequency': True,
 'date_added': datetime.datetime(2025, 6, 22, 0, 0),
 'routes': 1,
 'allsigshaveroute': True,
 'date_study_modified': None,
 'sigs': 2,
 'allsigshavecyclesigs': True,
 'allsigshaveschedule': True,
 'date_tracer_modified': None,
 'temp': None,
 'allsigshavesequence': False,
 'fullyspecified': False,
 'timings': 0,
 'id': 23050,
 'allsigshavedose': True,
 'blob_version': 2,
 'portions': 0,
 'allsigshavedoseunit': True,
 'branches': 0,
 'regimen': 'Cisplatin and Gemcitabine (GC)',
 'variant': 'Variant #41',
 'allsigshaveduration': False,
 'components': 2,
 'regimen_cui': 2570,
 'variant_cui': 130863,
 'allsigshavedurationunit': False,
 'cyclesigs': 1,
 'version': 1,
 'study_items': [<hemonc_import.final_model.entities.variants_StudyMap at 0x7fb7b5267000>,
 'tracer_items': [],
 'component_sigs': [<hemonc_import.final_model.entities.Sigs at 0x7fb7b5218a50>,
 'blob_items': [<hemonc_i

In [51]:
for sig in missing_vars[0].component_sigs:
    print(f"{sig.component} {sig.study}")

Cisplatin Fan et al. 2012|GLOB3|TAX 326
Docetaxel Fan et al. 2012|GLOB3|TAX 326


In [52]:
missing_vars[0].study_items[0].study_objects[0].__dict__ # Breast cancer study????

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fb7b548c350>,
 'protocol': False,
 'study_design': <Studies_Study_designEnum.IN_TO_CLASS_SWITCH: 'in-class switch'>,
 'id': 2486,
 'pubs_in_hemonc': 1,
 'study_design_imputed': False,
 'condition': 'Breast cancer',
 'reg_study': False,
 'temp': None,
 'condition_cui': 572.0,
 'registry': None,
 'trial_id': None,
 'date_added': datetime.datetime(2019, 5, 27, 0, 0),
 'sact': True,
 'unreg_study': False,
 'end': 'Not reported',
 'sponsor_type': <Studies_Sponsor_typeEnum.GOVERNMENT: 'government'>,
 'biomarker': 'Triple-negative',
 'enrollment': 'Not reported',
 'start': 'Not reported',
 'phase': 'Phase 3',
 'study': 'Fan et al. 2012',
 'study_group_items': [],
 'condition_obj': <hemonc_import.final_model.entities.Conditions at 0x7fb7b546c2b0>,
 'condition_cui_obj': <hemonc_import.final_model.entities.Conditions at 0x7fb7b546c2b0>,
 'sponsor_items': [<hemonc_import.final_model.entities.studies_SponsorMap at 0x7fb7b546c640>]}

In [53]:
for study_item in missing_vars[0].study_items:
    print(study_item.study)
    for study_object in study_item.study_objects:
        print(study_object.condition)
        print(study_object.condition_cui)

Fan et al. 2012
Breast cancer
572.0


In [54]:
for sig in missing_vars[-1].component_sigs:
    print(f"{sig.component} {sig.study}")

Cisplatin GEM20110714|JUPITER-02|WJTOG0203|Liu et al. 2024
Gemcitabine GEM20110714|JUPITER-02|WJTOG0203|Liu et al. 2024


In [55]:
for study_item in missing_vars[-1].study_items:
    print(study_item.study)
    for study_object in study_item.study_objects:
        print(study_object.condition)
        print(study_object.condition_cui)

CAPTAIN-1st
Nasopharyngeal carcinoma
639.0
GEM20110714
Nasopharyngeal carcinoma
639.0
JUPITER-02
Nasopharyngeal carcinoma
639.0
RATIONALE 309
Nasopharyngeal carcinoma
639.0


In [56]:
missing_vars[0].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fb7b52044d0>,
 'allsigshavefrequency': True,
 'date_added': datetime.datetime(2024, 10, 18, 0, 0),
 'routes': 1,
 'allsigshaveroute': True,
 'date_study_modified': None,
 'sigs': 2,
 'allsigshavecyclesigs': True,
 'allsigshaveschedule': True,
 'date_tracer_modified': datetime.datetime(2025, 3, 25, 0, 0),
 'temp': None,
 'allsigshavesequence': False,
 'fullyspecified': False,
 'timings': 0,
 'id': 4182,
 'allsigshavedose': True,
 'blob_version': 2,
 'portions': 0,
 'allsigshavedoseunit': True,
 'branches': 0,
 'regimen': 'Cisplatin and Docetaxel (DC)',
 'variant': 'Variant #01',
 'allsigshaveduration': False,
 'components': 2,
 'regimen_cui': 2569,
 'variant_cui': 130670,
 'allsigshavedurationunit': False,
 'cyclesigs': 1,
 'version': 1,
 'study_items': [<hemonc_import.final_model.entities.variants_StudyMap at 0x7fb7b53cd750>],
 'tracer_items': [<hemonc_import.final_model.entities.variants_TracerMap at 0x7fb7b5218690>],
 '

In [57]:
# with so.Session(engine) as session:
#     stmt_full = (
#         sa.select(models.Variants)
#         .join(subquery, models.Variants.variant_cui == subquery.c.variant_cui)
#     )
#     chosen_variants = session.execute(stmt_full).unique().scalars().all()

## Working on ~~semi-~~integrated pathway

In [58]:
from scoop.utils.treatment_selection import get_sig_component_stmt, categorise_components

In [59]:
components = get_sig_component_stmt(condition_cuis, models)
categorised = categorise_components(components)

In [60]:
components

In [61]:
with so.Session(engine) as session:
    display(pd.DataFrame(session.execute(sa.select(categorised))))

,variant_cui,canmed_major_class,canmed_minor_class,main_class,component,component_cui,study,phase,broad_category
0,129610.0,NaN,NaN,KRAS inhibitor,Adagrasib,45753.0,KRYSTAL-1_CRC|KRYSTAL-1_NSCLC,NaN,chemotherapy
1,129627.0,Microtubule inhibitor,Taxane,Taxane,Paclitaxel,379.0,LUX-Lung 5,NaN,chemotherapy
2,129627.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Lung 5,NaN,chemotherapy
3,129628.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Head and Neck 1|LUX-Head and Neck 3|LUX-Lu...,NaN,chemotherapy
4,129629.0,Tyrosine kinase inhibitor,EGFR inhibitor,EGFR inhibitor,Afatinib,13.0,LUX-Lung 2|LUX-Lung 3|LUX-Lung 6|LUX-Lung 7,NaN,chemotherapy
...,...,...,...,...,...,...,...,...,...
746,NaN,NaN,NaN,NaN,External beam radiotherapy,1608.0,Neuhaus et al. 2009|ENTER,NaN,radiotherapy
747,NaN,NaN,NaN,NaN,Granulocyte colony-stimulating factor,45734.0,ARCTIC|E7389-G000-302|TAX 320,NaN,supportive
748,NaN,NaN,NaN,NaN,Steroid,45523.0,BEYOND|CA184-041_NSCLC|CA184-104|Stathopoulos ...,NaN,supportive
749,NaN,NaN,NaN,NaN,Steroid,45523.0,CA184-041_NSCLC|CA184-041_SCLC,NaN,supportive


In [62]:
eg_comp_reqs = source_phases["neoadjuvant_chemo_ici"].component_requirements

In [63]:
eg_source_phase = source_phases["adjuvant_optchemo_alectinib"]

In [64]:
eg_comp_reqs = eg_source_phase.component_requirements

In [65]:
phase_type = eg_source_phase.type.title()

In [66]:

from scoop.utils.treatment_selection import (
    get_siggroup_identifying_subquery, subquery_sigs_valid_groups, 
    get_base_sigs_df, subset_df_on_phase, 
    )

In [67]:

valid_groups_subquery = get_siggroup_identifying_subquery(eg_comp_reqs, categorised)
sigs_for_valid_groups_subquery= subquery_sigs_valid_groups(valid_groups_subquery, models)
stmt = sa.select(sigs_for_valid_groups_subquery, models.Sigs).join(models.Sigs, models.Sigs.id==sigs_for_valid_groups_subquery.c.id) #.filter(or_for_piped_col(sigs_for_valid_groups_subquery.c.phase, phase_type.title()))


In [68]:
# to_col = phase_type.title()
# from_col = sigs_for_valid_groups_subquery.c.phase
# filtered_stmt = stmt.filter(or_for_piped_col(from_col, to_col))

In [69]:
with so.Session(engine) as session:
    new_cat_test = pd.DataFrame(session.execute(stmt))

In [70]:
eg_comp_reqs

ComponentRequirements(component_lists=[['alectinib']], category_restrictions={'chemotherapy': CategoryRestriction(type='min', amount=1), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)})

Note variant cui 131619 is for dexamethasone and wbrt

woooo :) ready to populate treatment phase

In [71]:
new_cat_test

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,AF-001JP,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
1,ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
2,J-ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
3,ALESIA,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
4,ALEX,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
5,ALINA,None,141790.0,141790.0,None,ALINA,Alectinib,771,Alectinib monotherapy,Variant #04,<hemonc_import.final_model.entities.Sigs objec...


In [72]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study", "regimen"], sort=False)["Sigs"].agg(list).to_dict()

{('None',
  129634.0,
  'AF-001JP|ALEX|J-ALEX',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7fb7b5327250>, <hemonc_import.final_model.entities.Sigs at 0x7fb7b5327250>, <hemonc_import.final_model.entities.Sigs at 0x7fb7b5327250>],
 ('None',
  129635.0,
  'ALESIA|ALEX',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7fb7b5326710>, <hemonc_import.final_model.entities.Sigs at 0x7fb7b5326710>],
 ('None',
  141790.0,
  'ALINA',
  'Alectinib monotherapy'): [<hemonc_import.final_model.entities.Sigs at 0x7fb7b5326850>]}

In [73]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].agg(list).to_list()

[[<hemonc_import.final_model.entities.Sigs at 0x7fb7b5327250>,

In [74]:
new_cat_test.groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].apply(list)

Series([], Name: Sigs, dtype: object)

In [75]:
new_cat_test.fillna("None").groupby(["phase", "variant_cui", "study"], sort=False)["Sigs"].agg(list).to_list()

[[<hemonc_import.final_model.entities.Sigs at 0x7fb7b5327250>,

In [76]:
new_cat_test # woooo - var test and non var test passed

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,AF-001JP,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
1,ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
2,J-ALEX,None,129634.0,129634.0,None,AF-001JP|ALEX|J-ALEX,Alectinib,768,Alectinib monotherapy,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
3,ALESIA,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
4,ALEX,None,129635.0,129635.0,None,ALESIA|ALEX,Alectinib,769,Alectinib monotherapy,Variant #02,<hemonc_import.final_model.entities.Sigs objec...
5,ALINA,None,141790.0,141790.0,None,ALINA,Alectinib,771,Alectinib monotherapy,Variant #04,<hemonc_import.final_model.entities.Sigs objec...


In [77]:
len(new_cat_test)

6

In [78]:
# fix source phase conditions DONE
# add in-statement study filtering

# add post-df study filtering
# add post-df phase filtering DONE
# add post-df context filtering

In [79]:
eg_source_phase = source_phases["definitive_rt"]

In [80]:
with so.Session(engine) as session:
    test_base_df = get_base_sigs_df(eg_source_phase, session)

In [81]:
subset_df_on_phase(test_base_df, eg_source_phase.type)

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
2,Ball et al. 1997,Definitive,NaN,NaN,Definitive,Ball et al. 1997,External beam radiotherapy,21487,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
3,CALGB 9130,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
4,Groen et al. 2004,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
5,JCOG0301,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
6,Morton et al. 1991,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
7,Perez et al. 1988,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
8,RTOG 93-04,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
9,LAMP,Definitive,NaN,NaN,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
10,RTOG 9410,Definitive,NaN,NaN,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
11,LAMP,Definitive,NaN,NaN,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21504,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...


In [82]:
test_base_df.drop_duplicates(subset=["id"])

,single_study,matched_phase,matched_variant_cui,variant_cui,phase,study,component,id,regimen,variant,Sigs
0,QUARTZ,NaN,131619.0,131619.0,NaN,QUARTZ,External beam radiotherapy,11162,Dexamethasone and WBRT,Variant #01,<hemonc_import.final_model.entities.Sigs objec...
1,ECOG E3590,Adjuvant,NaN,NaN,Adjuvant,ECOG E3590,External beam radiotherapy,21455,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
2,Ball et al. 1997,Definitive,NaN,NaN,Definitive,Ball et al. 1997,External beam radiotherapy,21487,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
3,CALGB 9130,Definitive,NaN,NaN,Definitive,CALGB 9130|Groen et al. 2004|JCOG0301|Morton e...,External beam radiotherapy,21489,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
9,LAMP,Definitive,NaN,NaN,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21499,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
11,LAMP,Definitive,NaN,NaN,Definitive,LAMP|RTOG 9410,External beam radiotherapy,21504,Radiation therapy,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...
13,ENTER,NaN,NaN,NaN,NaN,Neuhaus et al. 2009|ENTER,External beam radiotherapy,23903,Whole brain irradiation,N/A - radiation,<hemonc_import.final_model.entities.Sigs objec...


In [83]:
# Worked :)
# dfs = dict()
# for phase in source_phases.values():
#     with so.Session(engine) as session:
#         test_base_df = get_base_sigs_df(phase, session)
#         dfs[phase.source_phase_id] = subset_df_on_phase(test_base_df, phase.type)

In [84]:
from scoop.utils.treatment_selection import source_phase_to_sigdict

In [85]:
# Worked :) 
# for phase in source_phases.values():
#     with so.Session(engine) as session:
#         sigdict = source_phase_to_sigdict(phase, session)
#         print(f"{phase.source_phase_id}: len: {len(sigdict)}")

In [86]:
with so.Session(engine) as session:
    eg_sigdict = source_phase_to_sigdict(source_phases["neoadjuvant_chemo_ici"], session)

In [87]:
# Need to check these: non curative is a property of the STUDY not of the sigs
# Correction: per hemonckb data dictionary study context is a property of the study-regimen dyad
# Will need to think about implications of this but not immediate priority

# FIGURED IT OUT:
# This is happenning because supportive components are not assigned a variant CUI
# Thereby cleaving them from the primary components they match with regimen and study on
# Therefore even though the regimen might be e.g. cisplatin and pemetrexed
# These supportive components were not associated with those chemos in the grouping process
# dfs["best_supportive_care"]

In [88]:
source_phases["platinum_doublet_3rd_gen_agent"]

SourcePhaseYaml(source_phase_id='platinum_doublet_3rd_gen_agent', condition_cuis=[46104, 642], component_requirements=ComponentRequirements(component_lists=[['Platinum agent'], ['taxane', 'gemcitabine', 'vinorelbine']], category_restrictions={'chemotherapy': CategoryRestriction(type='exact', amount=2), 'radiotherapy': CategoryRestriction(type='exact', amount=0), 'immunotherapy': CategoryRestriction(type='exact', amount=0), 'surgery': CategoryRestriction(type='exact', amount=0)}), optional_reqs=None, curative_intent=True, regimens=None, type='None')

In [89]:
from scoop.utils.treatment_containers import SigBundle

In [90]:
def get_sigbundle_list(sigdict):
    sigbundle_list = []
    for (phase, variant_cui, study, regimen, variant_name), siglist in sigdict.items():
        try:
            variant_cui = int(variant_cui)
        except ValueError:
            variant_cui=None

        # if not isinstance(variant_cui, int):
        #     variant_cui = None
        if variant_name:
            name = regimen + " " + str(variant_name)
        else:
            name = regimen
        components = {sig.component for sig in siglist}
        sigbundle_list.append(SigBundle(name=name, sigs=siglist, 
                                        variant_cui=variant_cui,
                                        regimen=regimen, components=components))
    return sigbundle_list

In [91]:
get_sigbundle_list(eg_sigdict)[0]

Cisplatin and Docetaxel (DC) and Durvalumab Variant #01 (CUI: 130688)
Cisplatin: 100 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4936
Docetaxel: 85 mg/m^2 IV once on day(s) 1 in cycle(s) 1,2,3. Neoadjuvant SigID: 4937
Durvalumab: 750 mg IV once on day(s) 1 in cycle(s) 4,5. Neoadjuvant SigID: 4938

In [92]:
# for sb in get_sigbundle_list(eg_sigdict):
#     print(sb.__repr__)

## Working on protocols and combined phases

In [93]:
eg_protocol = protocols["neoadj_or_peri_ici_w_chemo"]

In [94]:
eg_protocol

ProtocolYaml(protocol_id='neoadj_or_peri_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], phases_for_removal=None, optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}})

In [95]:
phase_unions

{'met_intervention': PhaseUnionYaml(phase_id='met_intervention', source_phase_ids=['met_tki', 'met_ptdb', 'met_ptdb_w_ici'], remove=['met_tki', 'met_ptdb', 'met_ptdb_w_ici']),
 'ici_monotherapy_or_ici_with_chemotherapy_squamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_squamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_squamous'], remove=None),
 'ici_monotherapy_or_ici_with_chemotherapy_nonsquamous': PhaseUnionYaml(phase_id='ici_monotherapy_or_ici_with_chemotherapy_nonsquamous', source_phase_ids=['ici_monotherapy', 'ici_w_chemo_nonsquamous'], remove=None),
 'chemotherapy_or_atezolizumab_squamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_squamous', source_phase_ids=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab'], remove=['ptdb_squamous', 'single_chemo_squamous', 'atezolizumab']),
 'chemotherapy_or_atezolizumab_nonsquamous': PhaseUnionYaml(phase_id='chemotherapy_or_atezolizumab_nonsquamous', source_phase_ids=['ptdb_nonsquamous'

In [96]:
# TODO: add seq crt study pre-subsetting
# TODO: handle phase combinations
def get_source_phases_sb_list(protocol_yaml, source_phases, session, phase_unions={}):
    source_phases_sb_list = dict()
    for phase_id in eg_protocol.source_phase_ids:
        if phase_id in phase_unions.keys():
            phase_list = []
            for phase_id in phase_unions[phase_id].source_phase_ids:
                sigdict = source_phase_to_sigdict(source_phases[phase_id], session)
                phase_list += get_sigbundle_list(sigdict)
            source_phases_sb_list[phase_id] = phase_list
        else:
            sigdict = source_phase_to_sigdict(source_phases[phase_id], session)
            source_phases_sb_list[phase_id] = get_sigbundle_list(sigdict)
    return source_phases_sb_list

In [97]:
from scoop.utils.treatment_containers import TreatmentPhase, Protocol
from typing import Dict, List

In [98]:

def create_phases_list(protocol_yaml, restricted_sb_list: Dict[str, list], 
                       source_phases, phase_unions={}, full_sb_list={}, exclude={}) -> List[TreatmentPhase]:
    """
    Creates a list of phases suitable for populating protocols with.
    
    If creating an unrestricted protocol just pass the full_sb_list to 
    restricted_sb_list and ignore the full_sb_list parameter
    """
    phase_index = 0
    phases = []
    for source_phase_id in protocol_yaml.source_phase_ids:
        if source_phase_id in exclude:
            continue
        phase_index +=1
        if source_phase_id in restricted_sb_list.keys():
            sb_list = restricted_sb_list[source_phase_id]
        else:
            sb_list = full_sb_list[source_phase_id]
        if source_phase_id in phase_unions.keys():
            conditions = list({condition for source_phase_id in phase_unions[source_phase_id].source_phase_ids 
                          for condition in source_phases[source_phase_id].condition_cuis})
            categories = {category for source_phase_id in phase_unions[source_phase_id].source_phase_ids 
                          for category in source_phases[source_phase_id].categorise()}
        else:
            conditions = source_phases[source_phase_id].condition_cuis
            categories = source_phases[source_phase_id].categorise()
        treatment_phase = TreatmentPhase(index = phase_index, label = source_phase_id, 
                            options=sb_list, conditions=conditions, categories=categories)
        phases.append(treatment_phase)
    return phases

In [99]:
def create_protocols_by_shared_component(protocol_yaml, components_in_optional: set, sb_lists_by_phase: Dict[str, list], 
                                         req_sharing_phase: str, optional_phase:str):
    mutable_components = components_in_optional.copy()
    protocols = []
    for component in mutable_components.copy():
        component = mutable_components.pop()
        component_restricted_protocol= create_component_restricted_protocol(
            protocol_yaml, component, sb_lists_by_phase, req_sharing_phase, optional_phase
            )
        protocols.append(component_restricted_protocol)

    no_opt_phase_phases = create_phases_list(protocol_yaml, source_phases_sb_list, 
                                             source_phases, exclude={optional_phase})
    no_opt_phase_protocol = Protocol(name=protocol_yaml.protocol_id, phases=no_opt_phase_phases, 
                                    requirement_type="shared component",
                                    requirement_element=None
                                    )
    protocols.append(no_opt_phase_protocol)
    return protocols

In [100]:
def create_component_restricted_protocol(protocol_yaml, component: str, sb_lists_by_phase: Dict[str, list], 
                                         req_sharing_phase: str, optional_phase:str):
    req_sharing_phase_w_component = [sb for sb in source_phases_sb_list[req_sharing_phase] 
                                if component in sb.components]
    optional_phase_w_component = [sb for sb in source_phases_sb_list[optional_phase] 
                            if component in sb.components]
    with_components_sb_lists_by_phase = {req_sharing_phase: req_sharing_phase_w_component, 
                            optional_phase: optional_phase_w_component}
    with_component_phases = create_phases_list(protocol_yaml, with_components_sb_lists_by_phase, 
                                            source_phases, full_sb_list=source_phases_sb_list)
    component_restricted_protocol = Protocol(name=protocol_yaml.protocol_id, phases=with_component_phases, 
                                        requirement_type="shared component",
                                        requirement_element=component
                                        )
    return component_restricted_protocol

In [101]:
protocols

{'port_adjuvant_osimertinib': ProtocolYaml(protocol_id='port_adjuvant_osimertinib', source_phase_ids=['surgery_only', 'port', 'adj_osimertinib'], phases_for_removal=['adj_osimertinib', 'port'], optional_phases=None),
 'surgery_then_port': ProtocolYaml(protocol_id='surgery_then_port', source_phase_ids=['surgery_only', 'port'], phases_for_removal=['port'], optional_phases=None),
 'neoadj_or_peri_ici_w_chemo': ProtocolYaml(protocol_id='neoadj_or_peri_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], phases_for_removal=None, optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}}),
 'adjuvant_cisplatin_osimertinib': ProtocolYaml(protocol_id='adjuvant_cisplatin_osimertinib', source_phase_ids=['adjuvant_cisplatin_based', 'adjuvant_osimertinib'], phases_for_removal=None, optional_phases=None),
 'conc_crt_followed_by_durvalumab': ProtocolYaml(protocol_id='conc_crt_followed_by_durvalumab'

In [102]:
eg_protocol = protocols["neoadj_or_peri_ici_w_chemo"]

In [103]:
with so.Session(engine) as session:
    source_phases_sb_list = get_source_phases_sb_list(eg_protocol, source_phases, session, phase_unions)

In [104]:
def create_protocols_list(protocol_yaml, source_phases_sb_list, source_phases, phase_unions):
    if eg_protocol.optional_phases:
        optional_phase = list(protocol_yaml.optional_phases.keys())[0]
        requirement_type = protocol_yaml.optional_phases[optional_phase]["criteria"]
        is_shared_component_req = requirement_type.shared.lower()=="component"
        if is_shared_component_req:
            req_sharing_phase = requirement_type.as_phase
            optional_phase_sigs = source_phases_sb_list[optional_phase]
            components_in_optional = {component for sb in optional_phase_sigs for component in sb.components }
            pb = create_protocols_by_shared_component(protocol_yaml, components_in_optional, source_phases_sb_list, req_sharing_phase, optional_phase)
    else:
        phases = create_phases_list(protocol_yaml, source_phases_sb_list, source_phases, phase_unions)
        pb = [Protocol(name=protocol_yaml.protocol_id, phases=phases)]
    return pb

In [105]:
create_protocols_list(eg_protocol, source_phases_sb_list, source_phases, phase_unions)[0]

Treatment Protocol: neoadj_or_peri_ici_w_chemo (3 phase(s))

In [106]:
eg_protocol

ProtocolYaml(protocol_id='neoadj_or_peri_ici_w_chemo', source_phase_ids=['neoadjuvant_chemo_ici', 'surgery_only', 'adjuvant_ici'], phases_for_removal=None, optional_phases={'adjuvant_ici': {'criteria': OptionalPhaseCriteria(shared='component', as_phase='neoadjuvant_chemo_ici')}})

## ...

In [107]:
# First process all source phases into their phase type. Each should have an options_list
# Then process the union combinations - concatenate the options lists
# Then process the protocols - protocols should have phase_list attribute
# Then process the remaining not-removed phases into single-phase protocols.